PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [1]:
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import xgboost as xgb
import numpy as np
import pandas as pd
import random
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from tqdm import tqdm


import warnings
warnings.filterwarnings('ignore')  # Suppress Python warnings


random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [29]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch


In [34]:

fifa_df= pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Fifa/Data/fifa_stats_fully_processed.csv'))
elo_df = pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/ClubELO/Data/understat_team_elo_with_seasonal_elo.csv'))
aggregated_df = pd.read_csv('Data/sofa_team_aggregated.csv')


In [10]:
aggregated_df.columns

Index(['id', 'sofa_id', 'team_id', 'date', 'Corner kicks', 'Fouls',
       'Shots off target', 'Throw-ins', 'Ground duels', 'Aerial duels',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Game dominance', 'Big chances', 'Fouled in final third', 'Tackles won',
       'Dribbles', 'Crosses', 'Long balls', 'Yellow cards', 'Offsides',
       'Goalkeeper saves', 'h_a', 'Corner kicks_mean',
       'Corner kicks_mean_error', 'Fouls_mean', 'Fouls_mean_error',
       'Shots off target_mean', 'Shots off target_mean_error',
       'Throw-ins_mean', 'Throw-ins_mean_error', 'Ground duels_mean',
       'Ground duels_mean_error', 'Aerial duels_mean',
       'Aerial duels_mean_error', 'Blocked per shots on target_median',
       'Blocked per shots on target_median_error',
       'Shots on target per total shots_mean',
       'Shots on target per total shots_mean_error',
       'Hit woodwork per 

In [11]:
# Don't include errors
aggregated_df = aggregated_df[['id', 'team_id', 'date', 'Corner kicks', 'Fouls',
       'Shots off target', 'Throw-ins', 'Ground duels', 'Aerial duels',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
       'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves',
       'Game dominance',
       'h_a', 'Corner kicks_mean','Fouls_mean',
       'Game dominance_mean',
       'Shots off target_mean',
       'Throw-ins_mean',
       'Ground duels_mean','Aerial duels_mean',
       'Blocked per shots on target_median',
       
       'Shots on target per total shots_mean',
       
       'Hit woodwork per total shots_median',
       
       'Shots outside box per total shots_mean',
       'Big chances_mean',
       'Fouled in final third_mean',
       'Tackles won_mean',
       'Dribbles_mean',
       'Crosses_mean', 'Long balls_median',
       'Yellow cards_mean',
       'Offsides_median',
       'Goalkeeper saves_mean']]

In [13]:
aggregated_features_list= ['Corner kicks', 'Fouls',
       'Shots off target', 'Throw-ins', 'Ground duels', 'Aerial duels',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
       'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves',
       'Game dominance',
       'Corner kicks_mean','Fouls_mean',
       
       'Shots off target_mean',
       'Throw-ins_mean',
       'Ground duels_mean','Aerial duels_mean',
       'Blocked per shots on target_median',
       'Game dominance_mean',
       'Shots on target per total shots_mean',
       
       'Hit woodwork per total shots_median',
       
       'Shots outside box per total shots_mean',
       'Big chances_mean',
       'Fouled in final third_mean',
       'Tackles won_mean',
       'Dribbles_mean',
       'Crosses_mean', 'Long balls_median',
       'Yellow cards_mean',
       'Offsides_median',
       'Goalkeeper saves_mean']

In [14]:
elo_df = elo_df[['id', 'Elo_team_h',  'Elo_team_a', 'Seasonal_Elo_Home',
       'Seasonal_Elo_Away']].copy()
elo_df["ClubELO"] = elo_df["Elo_team_h"] - elo_df["Elo_team_a"]
elo_df["Seasonal_Elo"] = elo_df["Seasonal_Elo_Home"] - elo_df["Seasonal_Elo_Away"]
elo_df = elo_df[['id', 'ClubELO', 'Seasonal_Elo']].copy()

In [15]:
# Merge aggregated_df with elo_df, elo_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(aggregated_df, elo_df, on='id', how='left')
merged_df['ClubELO'] = np.where(merged_df['h_a'] == 'h', merged_df['ClubELO'], -merged_df['ClubELO'])
merged_df['Seasonal_Elo'] = np.where(merged_df['h_a'] == 'h', merged_df['Seasonal_Elo'], -merged_df['Seasonal_Elo'])

In [16]:
fifa_df.drop(columns=['yellow_card', 'red_card', 'key_passes', 'assists', 'xG',
       'xGBuildup'], inplace=True)

In [17]:
fifa_df.columns

Index(['id', 'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj'],
      dtype='object')

In [18]:
fifa_stats = ['Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']

In [19]:
# Merge merged DataFrame with fifa_df, fifa_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(merged_df, fifa_df, on='id', how='left')



In [20]:
merged_df[fifa_stats] = merged_df[fifa_stats].where(
    merged_df['h_a'] == 'h',
    -merged_df[fifa_stats]
)

In [21]:
def nan_analysis(df):
    """
    This function takes a DataFrame and returns a DataFrame with the percentage of NaN values for each column.
    Only columns with NaN values are included in the output DataFrame.
    """
    nan_percentage = df.isna().mean() * 100
    nan_percentage = nan_percentage[nan_percentage > 0].reset_index()
    nan_percentage.columns = ['Column Name', 'Percentage of NaN Values']
    nan_percentage = nan_percentage.sort_values(by='Percentage of NaN Values', ascending=False)
    return nan_percentage

In [22]:
nan_analysis(merged_df)

,Column Name,Percentage of NaN Values


In [23]:


# All inputs (choose either overalls or full FIFA features)
feature_cols = [
    # Aggregated values (example set, expand if needed)
    'Corner kicks_mean','Fouls_mean',
       'Shots off target_mean',
       'Throw-ins_mean',
       'Ground duels_mean','Aerial duels_mean',
       'Blocked per shots on target_median',
       
       'Shots on target per total shots_mean',
       'Game dominance_mean',
       'Hit woodwork per total shots_median',
       
       'Shots outside box per total shots_mean',
       'Big chances_mean',
       'Fouled in final third_mean',
       'Tackles won_mean',
       'Dribbles_mean',
       'Crosses_mean', 'Long balls_median',
       'Yellow cards_mean',
       'Offsides_median',
       'Goalkeeper saves_mean',

    # Elo
    "ClubELO", "Seasonal_Elo",

    #  full FIFA features
    'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']


In [24]:
def eval_random_forest(X_train, y_train, X_test, y_test, feature_cols):
    model = RandomForestRegressor(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    most_important_5_list = feature_importance_df.nlargest(5, 'importance').to_dict('records')
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), most_important_5_list

def eval_neural_net(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
              epochs=500, batch_size=32, callbacks=[es], verbose=0)
    
    preds = model.predict(X_test_scaled, verbose=0).ravel()
    model.scaler = scaler  # Store the scaler in the model for future use
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def eval_xgboost(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dte  = xgb.DMatrix(X_test, label=y_test)

    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    model = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dtr, 'train'), (dval, 'val')],
                      early_stopping_rounds=200, verbose_eval=False)
    preds = model.predict(dte, iteration_range=(0, model.best_iteration+1))
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def try_models(df, feature_cols, target, full=True, n_splits=5):
    X = df[feature_cols].values
    y = df[target].values
    tscv = TimeSeriesSplit(n_splits=n_splits)

    all_results = []
    splits = list(tscv.split(X))  # cache splits

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for name, func in [
            ("Random Forest", lambda X_tr, y_tr, X_te, y_te: eval_random_forest(X_tr, y_tr, X_te, y_te, feature_cols)),
            ("Neural Network", eval_neural_net),
            ("XGBoost", eval_xgboost)
        ]:
            mae, r2, top_features = func(X_train, y_train, X_test, y_test)
            all_results.append({
                'Fold': fold,
                'Target': target,
                'Model': name,
                'MAE': mae,
                'R2': r2,
                'Feature Types': 'Full' if full else 'Overall',
                'Important Features Dictionary': top_features
            })

    results_df = pd.DataFrame(all_results)

    perf_df = results_df.groupby(
        ["Target", "Model", "Feature Types"], as_index=False
    ).agg({
        "MAE": "mean",
        "R2": "mean",
        "Important Features Dictionary": lambda x: [d for d in x if d is not None]
    })

    # Train each model on the last fold for return values
    train_idx, test_idx = splits[-1]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Random Forest
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)

    # Neural Net
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    nn_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    nn_model.compile(optimizer='adam', loss='mean_squared_error')
    nn_es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    nn_model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
                 epochs=500, batch_size=32, callbacks=[nn_es], verbose=0)
    nn_model.scaler = scaler  # Store the scaler in the model for future use

    # XGBoost
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    xgb_model = xgb.train(
        xgb_params, dtr, num_boost_round=5000,
        evals=[(dtr, 'train'), (dval, 'val')],
        early_stopping_rounds=200, verbose_eval=False
    )

    return rf_model, nn_model, xgb_model, perf_df


In [25]:
performance_df = pd.DataFrame()
full_models_dict = {}

In [26]:
target_list= [
    'Corner kicks', 'Fouls', 'Game dominance',
       'Shots off target', 'Throw-ins', 'Ground duels', 'Aerial duels',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
       'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves',
]
for target in tqdm(target_list, desc="Processing targets"):
    rf_model, nn_model, xgb_model, perf_df = try_models(
        merged_df, feature_cols, target, full=True, n_splits=5
    )
    full_models_dict[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model,
        'XGBoost': xgb_model
    }

    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing targets: 100%|██████████| 20/20 [1:14:20<00:00, 223.01s/it]


In [27]:
performance_df["Important Features"]= "All"

In [28]:
performance_df

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,Corner kicks,Neural Network,Full,2.285348,0.003567,[],All
1,Corner kicks,Random Forest,Full,2.230650,0.071295,"[[{'feature': 'ClubELO', 'importance': 0.15072...",All
2,Corner kicks,XGBoost,Full,2.240685,0.066497,[],All
3,Fouls,Neural Network,Full,3.774776,-0.525064,[],All
4,Fouls,Random Forest,Full,3.445680,-0.223270,"[[{'feature': 'Penalties_adj', 'importance': 0...",All
5,Fouls,XGBoost,Full,3.660757,-0.419339,[],All
6,Game dominance,Neural Network,Full,0.071655,0.541230,[],All
7,Game dominance,Random Forest,Full,0.069530,0.568460,"[[{'feature': 'ClubELO', 'importance': 0.48763...",All
8,Game dominance,XGBoost,Full,0.069417,0.570969,[],All
9,Shots off target,Neural Network,Full,2.052515,-0.060608,[],All


In [29]:
def return_best_row_for_all_targets(df, target_list):
    # Get the row with the maximum importance for the specified feature
    best_rows = pd.DataFrame()
    for target in target_list:
        best_row = df[df.Target == target].sort_values(by='MAE', ascending=True).head(1)
        best_rows = pd.concat([best_rows, best_row], ignore_index=True)
    best_rows = best_rows.reset_index(drop=True)
    return best_rows

In [30]:
return_best_row_for_all_targets(performance_df, target_list)

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,Corner kicks,Random Forest,Full,2.230650,0.071295,"[[{'feature': 'ClubELO', 'importance': 0.15072...",All
1,Fouls,Random Forest,Full,3.445680,-0.223270,"[[{'feature': 'Penalties_adj', 'importance': 0...",All
2,Game dominance,XGBoost,Full,0.069417,0.570969,[],All
3,Shots off target,Random Forest,Full,1.982259,0.024337,"[[{'feature': 'ClubELO', 'importance': 0.12169...",All
4,Throw-ins,Random Forest,Full,4.925988,-0.029452,"[[{'feature': 'ClubELO', 'importance': 0.04709...",All
5,Ground duels,Random Forest,Full,0.049712,-0.006043,"[[{'feature': 'Vision_adj', 'importance': 0.03...",All
6,Aerial duels,Random Forest,Full,0.079745,0.036749,"[[{'feature': 'Overall Shooting_adj', 'importa...",All
7,Blocked per shots on target,Neural Network,Full,0.699867,-0.050046,[],All
8,Shots on target per total shots,XGBoost,Full,0.126637,-0.008430,[],All
9,Hit woodwork per total shots,Neural Network,Full,0.036690,-0.016141,[],All


In [31]:
import numpy as np
import pandas as pd
from collections import defaultdict

def aggregate_rf_importances(perf_df, target, feature_type, model="Random Forest",
                             top_k=10, agg="sum", zero_fill_mean=False):
    """
    Aggregate per-fold RF top-k importances into one ranked table.

    agg: "sum" -> sum across folds
         "mean" -> mean across occurrences
         "mean_zero" -> mean across all folds, treating missing folds as zero
    zero_fill_mean: deprecated; use agg="mean_zero" instead
    """
    # Select the cell
    mask = (
        (perf_df['Target'] == target) &
        (perf_df['Feature Types'] == feature_type) &
        (perf_df['Model'] == model)
    )
    if not mask.any():
        raise ValueError("No matching row in perf_df for the given filters.")
    cell = perf_df.loc[mask, 'Important Features Dictionary'].values[0]

    # Count folds represented in the cell
    if isinstance(cell, list) and cell and isinstance(cell[0], list):
        n_folds = len(cell)
        nested_lists = cell
    else:
        # Already flattened to a single list (or None)
        n_folds = 1 if isinstance(cell, list) else 0
        nested_lists = [cell] if isinstance(cell, list) else []

    # Flatten to records [{'feature':..., 'importance':...}, ...]
    records = []
    for fold_list in nested_lists:
        if not isinstance(fold_list, list):
            continue
        for rec in fold_list:
            if isinstance(rec, dict) and 'feature' in rec and 'importance' in rec:
                records.append(rec)

    if not records:
        return pd.DataFrame(columns=['feature', 'score', 'occurrence', 'rank'])

    # Aggregate
    totals = defaultdict(float)
    counts = defaultdict(int)
    for rec in records:
        f = rec['feature']
        imp = float(rec['importance'])
        totals[f] += imp
        counts[f] += 1

    rows = []
    for f, total in totals.items():
        if agg == "sum":
            score = total
        elif agg == "mean":
            score = total / counts[f]
        elif agg == "mean_zero":
            # treat missing folds as zero (uses n_folds)
            score = total / max(n_folds, 1)
        else:
            raise ValueError("agg must be one of {'sum','mean','mean_zero'}")
        rows.append({'feature': f, 'score': score, 'occurrence': counts[f]})

    out = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    out['rank'] = np.arange(1, len(out) + 1)
    return out.head(top_k)


In [32]:
# Get the first 6 most important features for each target
important_features_summary = {}
for target in target_list:
    important_features_summary[target] = aggregate_rf_importances(
        performance_df, target, feature_type='Full', model='Random Forest',
        top_k=6, agg='sum'
    )

In [33]:
important_features_summary

{'Corner kicks':                feature     score  occurrence  rank
 0              ClubELO  0.630436           5     1
 1         Seasonal_Elo  0.244023           3     2
 2           Vision_adj  0.153869           5     3
 3  Overall Defence_adj  0.083261           3     4
 4  Overall Passing_adj  0.079957           2     5
 5          Agility_adj  0.055132           2     6,
 'Fouls':              feature     score  occurrence  rank
 0  Interceptions_adj  0.181838           5     1
 1        Agility_adj  0.157086           4     2
 2      Dribbling_adj  0.150401           4     3
 3      Reactions_adj  0.146513           4     4
 4        FK Acc._adj  0.140995           4     5
 5      Penalties_adj  0.042976           1     6,
 'Game dominance':                feature     score  occurrence  rank
 0              ClubELO  2.455699           5     1
 1           Vision_adj  0.221033           5     2
 2          Heading_adj  0.167629           5     3
 3  Overall Passing_adj  0.137128

In [34]:
top_features_dict = {
    target: important_features_summary[target]['feature'].tolist()
    for target in target_list
}

In [35]:
# --- Random Forest ---
def build_RF_small(df, target, top_features, test_size=0.2):
    X = df[top_features].values
    y = df[target].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    return rf, mae, r2

# --- Neural Network (fixed: no leakage) ---
def build_NN_small(df, target, top_features, test_size=0.2, val_size=0.2):
    X = df[top_features].values
    y = df[target].values
    # Hold-out test block
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    # Validation from train block (no shuffling)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=val_size, shuffle=False
    )

    scaler = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_val_s  = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_tr_s.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

    model.fit(
        X_tr_s, y_tr,
        validation_data=(X_val_s, y_val),
        epochs=300, batch_size=32,
        callbacks=[es], verbose=0
    )

    y_pred = model.predict(X_test_s, verbose=0).ravel()
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # Attach scaler for downstream inference
    model.scaler = scaler
    return model, mae, r2

# --- Orchestrator ---
def get_small_models(df, top_features, target, full=True, n_splits=5):
    X = df[top_features].to_numpy()
    y = df[target].to_numpy()
    tscv = TimeSeriesSplit(n_splits=n_splits)

    perf = []
    for fold, (tr, te) in enumerate(tscv.split(X), 1):
        rf, mae_rf, r2_rf = build_RF_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        nn, mae_nn, r2_nn = build_NN_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        perf += [
            dict(Target=target, Model='Random Forest', MAE=mae_rf, R2=r2_rf,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
            dict(Target=target, Model='Neural Network', MAE=mae_nn, R2=r2_nn,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
        ]

    # --- average over folds & match old format ---
    perf_df = pd.DataFrame(perf)

    avg_df = (
        perf_df
        .groupby(['Target', 'Model', 'Feature_Types'], as_index=False)
        .agg(MAE=('MAE', 'mean'), R2=('R2', 'mean'))
        .rename(columns={'Feature_Types': 'Feature Types'})
    )

    # keep the same features column as before
    avg_df['Important Features'] = [top_features] * len(avg_df)

    # same columns/order as the old return
    avg_df = avg_df[['Target', 'Model', 'MAE', 'R2', 'Feature Types', 'Important Features']]

    return rf, nn, avg_df

In [36]:
full_models_dict_small = {}
for target in tqdm(target_list, desc="Processing small models"):
    top_features = top_features_dict[target]
    rf_model, nn_model, perf_df = get_small_models(
        merged_df, top_features, target, full=True
    )
    full_models_dict_small[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model
    }
    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing small models: 100%|██████████| 20/20 [10:10<00:00, 30.53s/it]


In [37]:
best_df=return_best_row_for_all_targets(performance_df, target_list)

In [38]:
best_plan = {}
for target in target_list:
    best_row = best_df[best_df['Target'] == target].iloc[0]
    best_plan[target] = (best_row['Model'], "full" if best_row['Important Features'] == 'All' else "small")
    

In [39]:
best_plan

{'Corner kicks': ('Neural Network', 'small'),
 'Fouls': ('Random Forest', 'full'),
 'Game dominance': ('XGBoost', 'full'),
 'Shots off target': ('Random Forest', 'full'),
 'Throw-ins': ('Random Forest', 'full'),
 'Ground duels': ('Neural Network', 'small'),
 'Aerial duels': ('Random Forest', 'full'),
 'Blocked per shots on target': ('Neural Network', 'small'),
 'Shots on target per total shots': ('Neural Network', 'small'),
 'Hit woodwork per total shots': ('Neural Network', 'full'),
 'Shots outside box per total shots': ('Neural Network', 'small'),
 'Big chances': ('XGBoost', 'full'),
 'Fouled in final third': ('XGBoost', 'full'),
 'Tackles won': ('Neural Network', 'full'),
 'Dribbles': ('Neural Network', 'small'),
 'Crosses': ('Random Forest', 'full'),
 'Long balls': ('XGBoost', 'full'),
 'Yellow cards': ('Random Forest', 'full'),
 'Offsides': ('XGBoost', 'full'),
 'Goalkeeper saves': ('Random Forest', 'full')}

In [40]:
# --- user-provided artifacts assumed to exist ---
# merged_df
# feature_cols
# top_features_dict
# full_models_dict, full_models_dict_small




def _get_features_for(target, scope):
    if scope == "full":
        return feature_cols
    elif scope == "small":
        return top_features_dict[target]
    else:
        raise ValueError(f"Unknown scope {scope!r}")

def _get_model_for(target, model_name, scope):
    store = full_models_dict if scope == "full" else full_models_dict_small
    if target not in store or model_name not in store[target]:
        raise KeyError(f"Model for target={target}, model={model_name}, scope={scope} not found.")
    return store[target][model_name]

def _predict_with_model(model, X, feats=None):
    """Unified prediction for Keras, xgboost.Booster, sklearn-style models."""

    # Keras (Neural Network)
    if isinstance(model, tf.keras.Model):
        scaler = getattr(model, "scaler", None)
        if scaler is not None:
            X = scaler.transform(X)
        else:
            print("[WARN] Keras model has no attached scaler; using raw features.")
        return model.predict(X, verbose=0).ravel()

    # Native XGBoost Booster (from xgb.train)
    if isinstance(model, xgb.Booster):
        # If you want, you can pass feature_names=feats for extra safety:
        dX = xgb.DMatrix(X)  # or xgb.DMatrix(X, feature_names=feats)
        return model.predict(dX).ravel()

    # sklearn-style (RandomForestRegressor, XGBRegressor, etc.)
    if hasattr(model, "predict"):
        return np.asarray(model.predict(X)).ravel()

    raise TypeError(f"Unsupported model type: {type(model)}")

# --- build all {target}_agg columns ---
for target, (model_name, scope) in best_plan.items():
    feats = _get_features_for(target, scope)
    mdl   = _get_model_for(target, model_name, scope)

    # matrix in the SAME column order used in training
    X = merged_df[feats].to_numpy()

    # (optional) fail-fast NaN check
    if np.isnan(X).any():
        bad_cols = sorted(set(np.where(np.isnan(X))[1]))
        raise ValueError(
            f"NaNs in features for target '{target}': {[feats[i] for i in bad_cols]}. "
            "Impute or drop rows before predicting."
        )

    preds = _predict_with_model(mdl, X, feats)
    merged_df[f"{target}_agg"] = preds

cols_preview = [f"{t}_agg" for t in best_plan]
print("Created columns:", cols_preview)
print(merged_df[cols_preview].head())

Created columns: ['Corner kicks_agg', 'Fouls_agg', 'Game dominance_agg', 'Shots off target_agg', 'Throw-ins_agg', 'Ground duels_agg', 'Aerial duels_agg', 'Blocked per shots on target_agg', 'Shots on target per total shots_agg', 'Hit woodwork per total shots_agg', 'Shots outside box per total shots_agg', 'Big chances_agg', 'Fouled in final third_agg', 'Tackles won_agg', 'Dribbles_agg', 'Crosses_agg', 'Long balls_agg', 'Yellow cards_agg', 'Offsides_agg', 'Goalkeeper saves_agg']
   Corner kicks_agg  Fouls_agg  Game dominance_agg  Shots off target_agg  \
0          5.673473   3.366667            0.418490              4.656667   
1          5.190045   2.803333            0.483724              6.536667   
2          6.044593   2.413333            0.543084              4.043333   
3          5.201778   3.393333            0.338268              7.223333   
4          6.590044   3.426667            0.495292              7.830000   

   Throw-ins_agg  Ground duels_agg  Aerial duels_agg  \
0     

In [41]:
discrete_features = [
    'Corner kicks', 'Fouls', 'Shots off target', 'Throw-ins',
    'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
    'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves'
]
# round discrete features to nearest integer
for feat in discrete_features:
    agg_col = f"{feat}_agg"
    if agg_col in merged_df.columns:
        merged_df[agg_col] = merged_df[agg_col].round().astype(int)
# Save the final DataFrame with aggregated predictions
merged_df.to_csv('Data/sofa_team_model_aggregated.csv', index=False)

In [42]:
merged_df.columns

Index(['id', 'team_id', 'date', 'Corner kicks', 'Fouls', 'Shots off target',
       'Throw-ins', 'Ground duels', 'Aerial duels',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
       'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves',
       'Game dominance', 'h_a', 'Corner kicks_mean', 'Fouls_mean',
       'Game dominance_mean', 'Shots off target_mean', 'Throw-ins_mean',
       'Ground duels_mean', 'Aerial duels_mean',
       'Blocked per shots on target_median',
       'Shots on target per total shots_mean',
       'Hit woodwork per total shots_median',
       'Shots outside box per total shots_mean', 'Big chances_mean',
       'Fouled in final third_mean', 'Tackles won_mean', 'Dribbles_mean',
       'Crosses_mean', 'Long balls_median', 'Yellow cards_mean',
       'Offsides_median', 'Goalke

In [44]:
sofa_org= ['Corner kicks', 'Fouls', 'Shots off target',
       'Throw-ins', 'Ground duels', 'Aerial duels','Game dominance',
       'Blocked per shots on target', 'Shots on target per total shots',
       'Hit woodwork per total shots', 'Shots outside box per total shots',
       'Big chances', 'Fouled in final third', 'Tackles won', 'Dribbles',
       'Crosses', 'Long balls', 'Yellow cards', 'Offsides', 'Goalkeeper saves']
sofa_window= ['Corner kicks_mean', 'Fouls_mean', 'Shots off target_mean',
       'Throw-ins_mean', 'Ground duels_mean', 'Aerial duels_mean',
       'Blocked per shots on target_median','Game dominance_mean',
       'Shots on target per total shots_mean',
       'Hit woodwork per total shots_median',
       'Shots outside box per total shots_mean', 'Big chances_mean',
       'Fouled in final third_mean', 'Tackles won_mean', 'Dribbles_mean',
       'Crosses_mean', 'Long balls_median', 'Yellow cards_mean',
       'Offsides_median', 'Goalkeeper saves_mean']
sofa_agg= ['Corner kicks_agg',
       'Fouls_agg', 'Shots off target_agg', 'Throw-ins_agg',
       'Ground duels_agg', 'Aerial duels_agg',
       'Blocked per shots on target_agg','Game dominance_agg',
       'Shots on target per total shots_agg',
       'Hit woodwork per total shots_agg',
       'Shots outside box per total shots_agg', 'Big chances_agg',
       'Fouled in final third_agg', 'Tackles won_agg', 'Dribbles_agg',
       'Crosses_agg', 'Long balls_agg', 'Yellow cards_agg', 'Offsides_agg',
       'Goalkeeper saves_agg']

sofa={ "Corner kicks": ("Corner kicks_mean", "Corner kicks_agg"),
       "Fouls": ("Fouls_mean", "Fouls_agg"),
       "Shots off target": ("Shots off target_mean", "Shots off target_agg"),
       "Throw-ins": ("Throw-ins_mean", "Throw-ins_agg"),
       "Ground duels": ("Ground duels_mean", "Ground duels_agg"),
       "Aerial duels": ("Aerial duels_mean", "Aerial duels_agg"),
       "Blocked per shots on target": ("Blocked per shots on target_median", "Blocked per shots on target_agg"),
       "Game dominance": ("Game dominance_mean", "Game dominance_agg"),
       "Shots on target per total shots": ("Shots on target per total shots_mean", "Shots on target per total shots_agg"),
       "Hit woodwork per total shots": ("Hit woodwork per total shots_median", "Hit woodwork per total shots_agg"),
       "Shots outside box per total shots": ("Shots outside box per total shots_mean", "Shots outside box per total shots_agg"),
       "Big chances": ("Big chances_mean", "Big chances_agg"),
       "Fouled in final third": ("Fouled in final third_mean", "Fouled in final third_agg"),
       "Tackles won": ("Tackles won_mean", "Tackles won_agg"),
       "Dribbles": ("Dribbles_mean", "Dribbles_agg"),
       "Crosses": ("Crosses_mean", "Crosses_agg"),
       "Long balls": ("Long balls_median", "Long balls_agg"),
       "Yellow cards": ("Yellow cards_mean", "Yellow cards_agg"),
       "Offsides": ("Offsides_median", "Offsides_agg"),
       "Goalkeeper saves": ("Goalkeeper saves_mean", "Goalkeeper saves_agg")}

In [66]:
import re
import pandas as pd
from sklearn.metrics import mean_absolute_error

records = []  # collect rows here

for target in sofa_org:
    rolling_aggregation_type = re.search(r'_(mean|median)', sofa[target][0]).group(1)

    record = {
        "Feature": target,
        "Rolling Aggregation Method": rolling_aggregation_type,

        "Modeling Aggregation Method": best_plan[target][0],
        "Modeling Aggregation Type": best_plan[target][1],
            "Rolling Aggregation MAE": f"{mean_absolute_error(
            merged_df[target], merged_df[sofa[target][0]]
        ): .3f}",
        "Modeling Aggregation MAE": f"{mean_absolute_error(
            merged_df[target], merged_df[sofa[target][1]]
        ): .3f}",

        "MAE Improvement Percentage": f"{(mean_absolute_error(
            merged_df[target], merged_df[sofa[target][0]]
        ) - mean_absolute_error(
            merged_df[target], merged_df[sofa[target][1]]
        )) / mean_absolute_error(
            merged_df[target], merged_df[sofa[target][0]]
        )*100: .3f}%"
    }

    records.append(record)

# convert list of dicts into DataFrame
last_df = pd.DataFrame(records)


In [67]:
last_df

,Feature,Rolling Aggregation Method,Modeling Aggregation Method,Modeling Aggregation Type,Rolling Aggregation MAE,Modeling Aggregation MAE,MAE Improvement Percentage
0,Corner kicks,mean,Neural Network,small,2.403,2.130,11.346%
1,Fouls,mean,Random Forest,full,3.701,1.528,58.703%
2,Shots off target,mean,Random Forest,full,2.086,0.880,57.795%
3,Throw-ins,mean,Random Forest,full,5.168,2.250,56.452%
4,Ground duels,mean,Neural Network,small,0.051,0.049,3.647%
5,Aerial duels,mean,Random Forest,full,0.083,0.038,54.674%
6,Game dominance,mean,XGBoost,full,0.106,0.034,68.162%
7,Blocked per shots on target,median,Neural Network,small,0.678,0.687,-1.288%
8,Shots on target per total shots,mean,Neural Network,small,0.129,0.124,3.950%
9,Hit woodwork per total shots,median,Neural Network,full,0.026,0.037,-41.960%
